# What can 30,000 Instagram posts tell you?

The working behind the [findings report](../findings/README.md), using 29,999 posts from the Kaggle Instagram Analytics Dataset.

The calculations live in [`findings/analysis.py`](../findings/analysis.py) and the charts in [`findings/charts.py`](../findings/charts.py), drawn with [bezel-charts](https://pypi.org/project/bezel-charts/). This notebook calls the same functions, so its numbers always match the report.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "findings").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "findings"))

import analysis as an
import charts

## Look before analysing

One row per post, no missing values. Before asking what drives engagement, check what each column actually is.

In [ ]:
posts = an.posts()
print(f"{len(posts):,} posts, {posts.isna().sum().sum()} missing values")
posts.head()

## 1. The engagement rate is a formula

If the rate is just interactions divided by impressions, it isn't a separate measurement, and predicting it from those columns would be circular.

In [ ]:
an.formula()

Some posts have more interactions than impressions, which gives rates over 100%. They pull the average well above the typical post, so the rest of the analysis uses medians and rank-based tests.

In [ ]:
an.impossible()

In [ ]:
charts.rate_bands()

## 2. No format wins

The first version compared averages. Medians, with 95% bootstrap intervals, show the typical post in every format is level. The Kruskal–Wallis test asks whether the gaps are larger than chance.

In [ ]:
an.bootstrap_median("media_type", "engagement_rate").round(2)

In [ ]:
an.first_version_claims().round(3)

In [ ]:
charts.formats()

## 3. No category gains more followers

In [ ]:
an.bootstrap_median("content_category", "followers_gained")

In [ ]:
charts.categories()

## 4. Nothing is related to anything

On real accounts, reach and likes move together. Here every correlation is close to zero, except reach and impressions, which are built from each other.

In [ ]:
an.correlations().round(2)

In [ ]:
charts.reach_and_likes()

## 5. Fingerprints of generated data

Even spreads, one clock time on every post, and perfectly balanced groups. A high Kolmogorov–Smirnov p-value means a column is consistent with numbers drawn evenly at random.

In [ ]:
an.uniform_tests().round(3).to_frame("p-value against an even spread")

In [ ]:
an.fingerprints()

In [ ]:
charts.likes_spread()

## 6. Chance alone produces "findings"

24 group comparisons and 45 category pairs. Shuffling the categories shows how many "significant" pairs pure chance creates.

In [ ]:
an.all_group_tests().sort_values("p_value").head(6).round(3)

In [ ]:
an.chance()

In [ ]:
charts.chance()

## 7. The first model was circular

The first version's linear regression, rebuilt, then the same inputs on a log scale, then honest models that only use what's known before posting, trained on earlier posts and tested on later ones.

In [ ]:
model = an.models()
model["r2"].to_frame("R² on unseen posts")

In [ ]:
charts.model_scores()

## What this data can't tell you

Anything about Instagram. The numbers behave like independent random draws, so they're useful for practising code and as a baseline with no signal, not as evidence. The [findings report](../findings/README.md) sets this out in full.